# 04 — Fixing video-level leakage: dedup + GroupShuffleSplit

**Why this notebook exists**: `03_random_forest.ipynb` trained on a plain row-random 80/20 split and got a suspiciously large jump over the linear baseline (R² 0.097 → 0.776). Investigation found the row-random split was leaking: this dataset has one row per `(video_id, video_trending_country, trending_date)`, but the v2 feature set (`video_category_id`, `tag_count`, `title_length`, `title_has_caps_word`, `publish_hour`, `publish_dayofweek`) is entirely video-level — none of it varies by country or date. That means:

1. **Same video, same `trending_date`, different countries → identical `video_view_count`.** Confirmed directly: `ekr2nIex040` ("ROSÉ & Bruno Mars - APT.") shows the exact same view count (64,890,495) across all 91 countries it trended in on 2024-10-20 — it's a single global daily snapshot broadcast per country, not a country-specific count. These rows are 100% redundant given the v2 feature set (country isn't even a feature) — pure duplication, zero information.
2. **Same video, different `trending_date` → `video_view_count` genuinely differs**, climbing as the video accumulates more views during its trending run (e.g. the same APT. video: 143.7M on 2024-10-26 → 407.5M by 2024-11-23). These rows share identical v2 features but different targets — a row-random split scatters a video's early- and late-trending snapshots across both train and val, so the model can partly "recognize" a video from train and interpolate its val-set target, rather than generalizing to unseen videos.

**The fix, in two steps**:
1. **Dedup**: collapse same-day country duplicates down to one row per `(video_id, trending_date)` — no information lost, since those rows were exact duplicates in the v2 feature+target space.
2. **Group-split**: on what remains, use `GroupShuffleSplit` grouped by `video_id` so all of a given video's remaining (date-level) rows land entirely in train or entirely in val, never split across both.

Then rerun both the Linear Regression baseline and the Random Forest on the corrected data, same v2 feature set, for a clean comparison.

In [1]:
import sys
sys.path.append('../src')

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

from features import build_features

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

MODELS_DIR = Path('../models')

DATA_PATH = Path('../data/processed/train.parquet')
df = pd.read_parquet(DATA_PATH)
df = build_features(df)
print(f"Loaded and featurized {len(df):,} rows")

Loaded and featurized 9,876,866 rows


## Build model_df (same raw columns as before, plus `video_id` and `trending_date` for dedup/grouping)

In [2]:
FEATURE_COLUMNS_RAW = [
    'video_category_id',
    'tag_count',
    'title_length',
    'title_has_caps_word',
    'publish_hour',
    'publish_dayofweek',
    'video_trending_country',
]
FEATURE_COLUMNS_RAW_V2 = [c for c in FEATURE_COLUMNS_RAW if c != 'video_trending_country']
TARGET_COLUMN = 'video_view_count'
KEY_COLUMNS = ['video_id', 'trending_date']

model_df = df[FEATURE_COLUMNS_RAW + [TARGET_COLUMN] + KEY_COLUMNS].dropna(
    subset=FEATURE_COLUMNS_RAW + [TARGET_COLUMN]
).copy()
print(f"Rows before dedup: {len(model_df):,} (from {len(df):,})")

Rows before dedup: 9,876,866 (from 9,876,866)


## Step 1 — Dedup: one row per `(video_id, trending_date)`

Sorting by `video_trending_country` before `drop_duplicates(keep='first')` just makes the kept row deterministic/reproducible — since same-day rows across countries are exact duplicates in the v2 feature+target space, which country happens to be kept doesn't matter.

In [3]:
model_df_sorted = model_df.sort_values(['video_id', 'trending_date', 'video_trending_country'])
deduped_df = model_df_sorted.drop_duplicates(subset=['video_id', 'trending_date'], keep='first').copy()

print(f"Rows before dedup: {len(model_df):,}")
print(f"Rows after dedup:  {len(deduped_df):,}")
print(f"Dropped as country-duplicate rows: {len(model_df) - len(deduped_df):,} "
      f"({(len(model_df) - len(deduped_df)) / len(model_df):.1%})")
print(f"Unique video_id in deduped data: {deduped_df['video_id'].nunique():,}")
print(f"Mean rows per video after dedup (i.e. mean distinct trending dates/video): "
      f"{len(deduped_df) / deduped_df['video_id'].nunique():.2f}")

Rows before dedup: 9,876,866
Rows after dedup:  4,183,259
Dropped as country-duplicate rows: 5,693,607 (57.6%)
Unique video_id in deduped data: 1,039,987
Mean rows per video after dedup (i.e. mean distinct trending dates/video): 4.02


## Encode v2 features — identical column set to the earlier baseline/RF

In [4]:
X_raw_v2 = deduped_df[FEATURE_COLUMNS_RAW_V2]
y = deduped_df[TARGET_COLUMN].astype('float64')
groups = deduped_df['video_id']

X_encoded = pd.get_dummies(X_raw_v2, columns=['video_category_id'], drop_first=True)
FEATURE_COLUMNS = list(X_encoded.columns)
print(f"Feature matrix shape: {X_encoded.shape}")

FEATURES_PATH = MODELS_DIR / 'baseline_feature_columns.json'
with open(FEATURES_PATH) as f:
    saved_v2_columns = json.load(f)
assert FEATURE_COLUMNS == saved_v2_columns, 'Feature column mismatch vs. saved v2 baseline'
print('Confirmed: identical column set and order to the v2 baseline.')

Feature matrix shape: (4183259, 19)
Confirmed: identical column set and order to the v2 baseline.


## Step 2 — GroupShuffleSplit by `video_id`

`test_size=0.2, random_state=42` — same proportions as before, but now no `video_id` appears on both sides of the split.

In [5]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_encoded, y, groups=groups))

X_train, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
groups_train, groups_val = groups.iloc[train_idx], groups.iloc[val_idx]

print(f"Train: {X_train.shape} ({groups_train.nunique():,} unique videos)")
print(f"Val:   {X_val.shape} ({groups_val.nunique():,} unique videos)")
print(f"Video overlap between train and val: {len(set(groups_train) & set(groups_val)):,} (should be 0)")

Train: (3343308, 19) (831,989 unique videos)
Val:   (839951, 19) (207,998 unique videos)


Video overlap between train and val: 0 (should be 0)


## Train Linear Regression on the corrected split

In [6]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)

rmse_lr = np.sqrt(mean_squared_error(y_val, y_pred_lr))
r2_lr = r2_score(y_val, y_pred_lr)

print('=== Linear Regression (deduped + grouped) — validation results ===')
print(f'RMSE: {rmse_lr:,.2f} views')
print(f'R^2:  {r2_lr:.4f}')

=== Linear Regression (deduped + grouped) — validation results ===
RMSE: 7,152,346.60 views
R^2:  0.0458


## Train Random Forest on the corrected split

Same hyperparameters as the leaked run in `03_random_forest.ipynb` (`n_estimators=200, max_depth=15, n_jobs=-1, random_state=42`) — no retuning, so the before/after comparison isolates the effect of fixing the split, not a hyperparameter change.

In [7]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

rmse_rf = np.sqrt(mean_squared_error(y_val, y_pred_rf))
r2_rf = r2_score(y_val, y_pred_rf)

print('=== Random Forest (deduped + grouped) — validation results ===')
print(f'RMSE: {rmse_rf:,.2f} views')
print(f'R^2:  {r2_rf:.4f}')

=== Random Forest (deduped + grouped) — validation results ===
RMSE: 7,011,337.01 views
R^2:  0.0831


## Full comparison: leaked (row-random) vs. corrected (dedup + group-split)

In [8]:
rmse_lr_leaked = 18_932_915.0
r2_lr_leaked = 0.0972
rmse_rf_leaked = 9_429_240.85
r2_rf_leaked = 0.7761

print(f'{"":42s} {"RMSE":>18s} {"R^2":>10s}')
print(f'{"Linear Regression — row-random (leaked)":42s} {rmse_lr_leaked:>18,.2f} {r2_lr_leaked:>10.4f}')
print(f'{"Linear Regression — dedup + grouped":42s} {rmse_lr:>18,.2f} {r2_lr:>10.4f}')
print(f'{"Random Forest — row-random (leaked)":42s} {rmse_rf_leaked:>18,.2f} {r2_rf_leaked:>10.4f}')
print(f'{"Random Forest — dedup + grouped":42s} {rmse_rf:>18,.2f} {r2_rf:>10.4f}')
print()
print(f'Linear RMSE change: {rmse_lr - rmse_lr_leaked:+,.2f} ({(rmse_lr - rmse_lr_leaked) / rmse_lr_leaked:+.2%})')
print(f'Linear R^2 change:  {r2_lr - r2_lr_leaked:+.4f}')
print(f'RF RMSE change:     {rmse_rf - rmse_rf_leaked:+,.2f} ({(rmse_rf - rmse_rf_leaked) / rmse_rf_leaked:+.2%})')
print(f'RF R^2 change:      {r2_rf - r2_rf_leaked:+.4f}')

                                                         RMSE        R^2
Linear Regression — row-random (leaked)         18,932,915.00     0.0972
Linear Regression — dedup + grouped              7,152,346.60     0.0458
Random Forest — row-random (leaked)              9,429,240.85     0.7761
Random Forest — dedup + grouped                  7,011,337.01     0.0831

Linear RMSE change: -11,780,568.40 (-62.22%)
Linear R^2 change:  -0.0514
RF RMSE change:     -2,417,903.84 (-25.64%)
RF R^2 change:      -0.6930


## Save the corrected models

In [9]:
LR_PATH = MODELS_DIR / 'baseline_linear_regression_v2_dedup_grouped.pkl'
RF_PATH = MODELS_DIR / 'random_forest_dedup_grouped.pkl'

joblib.dump(lr, LR_PATH)
joblib.dump(rf, RF_PATH)
print(f'Saved Linear Regression to {LR_PATH}')
print(f'Saved Random Forest to {RF_PATH}')
print()
print('NOTE: models/baseline_linear_regression_v2.pkl and models/random_forest.pkl '
      '(the row-random, leaked-split versions) are left in place for reference/comparison, '
      'not overwritten. baseline_feature_columns.json is unchanged — same 19 v2 columns '
      'apply to both the leaked and corrected models.')

Saved Linear Regression to ../models/baseline_linear_regression_v2_dedup_grouped.pkl
Saved Random Forest to ../models/random_forest_dedup_grouped.pkl

NOTE: models/baseline_linear_regression_v2.pkl and models/random_forest.pkl (the row-random, leaked-split versions) are left in place for reference/comparison, not overwritten. baseline_feature_columns.json is unchanged — same 19 v2 columns apply to both the leaked and corrected models.


## Summary

**Dataset shrank by 57.6%** on dedup: 9,876,866 → 4,183,259 rows (dropped 5,693,607 same-day country-duplicate rows). The 1,039,987 unique videos average 4.02 distinct trending dates each post-dedup (down from 9.50 raw rows/video pre-dedup — confirming most of the original row count was country repetition, not date-level signal). GroupShuffleSplit then produced Train: 3,343,308 rows / 831,989 videos, Val: 839,951 rows / 207,998 videos, with confirmed zero video_id overlap between the two.

**Results, corrected split:**

| | RMSE | R² |
|---|---:|---:|
| Linear Regression — row-random (leaked) | 18,932,915 | 0.0972 |
| Linear Regression — dedup + grouped | 7,152,347 | 0.0458 |
| Random Forest — row-random (leaked) | 9,429,241 | 0.7761 |
| Random Forest — dedup + grouped | 7,011,337 | 0.0831 |

**Random Forest's leaked 0.776 R² was almost entirely the video-identity leakage — it collapses to 0.083 once no video can appear on both sides of the split.** That confirms the diagnosis: a depth-15 forest could memorize `(title_length, tag_count, category, hour, dow) → view_count` lookups for videos it had already seen in train, and 95.3% of the old validation rows had exactly that leak available. With leakage closed, Random Forest still modestly outperforms Linear Regression (R² 0.083 vs. 0.046, RMSE 7.01M vs. 7.15M) — a real but far more modest edge from capturing non-linear/interaction effects, not from memorization.

**RMSE fell for both models even though R² also fell — not a contradiction.** The old row-random validation set's target had std ≈ 19.93M (dominated by mega-viral videos whose ~9 country-duplicate rows each inflated their weight in both the split and its variance). The corrected validation set's target has std ≈ 7.32M — removing the country duplication removed most of that inflation, shrinking the achievable error (lower RMSE) but also shrinking the variance R² is measured against, which fell faster than the errors did (both models now explain a smaller share of a smaller, harder pie). This is the more honest number: predicting raw view count from five pre-publish, video-level fields, without knowing which day of a video's trending run is being sampled, is a genuinely hard problem — R² in the 0.05–0.08 range is a believable ceiling for this feature set, not the 0.78 the leaked split implied.

**Next step worth considering**: even post-fix, a video contributes one row per trending date with *identical* pre-publish features but a *growing* target (the same video's view count 3 weeks into trending vs. day 1 look wildly different for the same X) — this is irreducible label noise given the current feature set. Adding a "days since publish" / "day of trending run" feature, or restricting to one canonical snapshot per video (e.g. first trending day), would likely be the next real lever on R², more so than further model tuning on the current feature set.